In [2]:
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import os
from tqdm import tqdm

# --- 1. 설정: 경로 및 모델 ---

# CSV 파일 및 이미지 폴더 경로 설정
BASE_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data"
CSV_PATH = os.path.join(BASE_PATH, "dev_test.csv")
IMAGE_DIR = os.path.join(BASE_PATH, "input_images")
OUTPUT_PATH = os.path.join(BASE_PATH, "submission_abcd.csv") # 출력 파일명 변경

# 사용할 디바이스 설정 (GPU 우선 사용)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 사전 학습된 CLIP 모델 및 프로세서 로드
MODEL_ID = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_ID)

# --- 2. 데이터 로드 ---

try:
    df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_PATH}")
    exit()

# 결과를 저장할 리스트 초기화
results = []
# 인덱스를 알파벳으로 매핑하기 위한 리스트
ANSWER_MAP = ['A', 'B', 'C', 'D']

# --- 3. 모델 추론 ---

print("Starting inference...")
with torch.no_grad():
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            test_id = row['ID']
            img_filename = row['img_path'].split('/')[-1]
            img_path = os.path.join(IMAGE_DIR, img_filename)
            
            question = row['Question']
            choices = [row['A'], row['B'], row['C'], row['D']]

            image = Image.open(img_path)
            
            # 질문과 선택지를 조합하여 4개의 프롬프트 생성
            text_prompts = [f"{question} {choice}" for choice in choices]

            # 모델 입력 준비
            inputs = processor(text=text_prompts, images=image, return_tensors="pt", padding=True).to(device)
            
            # 모델 추론 실행
            outputs = model(**inputs)

            # 이미지-텍스트 유사도 점수 (logits) 추출
            logits_per_image = outputs.logits_per_image
            
            # 가장 높은 점수를 가진 선택지의 인덱스 찾기 (0, 1, 2, 3)
            prediction_index = torch.argmax(logits_per_image).item()
            
            # ⭐ 수정된 부분: 인덱스를 'A', 'B', 'C', 'D'로 변환
            answer = ANSWER_MAP[prediction_index]
            
            results.append({'ID': test_id, 'answer': answer})

        except Exception as e:
            print(f"Error processing row {index} (ID: {row.get('ID', 'N/A')}): {e}")
            results.append({'ID': row.get('ID', 'N/A'), 'answer': 'E'}) # 에러 발생 시 'E'로 표기

# --- 4. 제출 파일 생성 ---

print("Inference finished. Creating submission file...")

submission_df = pd.DataFrame(results)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"Submission file created successfully at: {OUTPUT_PATH}")
print("\n--- Top 5 predictions ---")
print(submission_df.head())
print("------------------------")

Using device: cuda
Starting inference...


100%|██████████| 60/60 [00:02<00:00, 28.91it/s]

Inference finished. Creating submission file...
Submission file created successfully at: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission_abcd.csv

--- Top 5 predictions ---
         ID answer
0  TEST_000      B
1  TEST_001      B
2  TEST_002      C
3  TEST_003      C
4  TEST_004      A
------------------------


In [3]:
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
import os
from tqdm import tqdm

# --- 1. 설정: 경로 및 모델 ---

# CSV 파일 및 이미지 폴더 경로 설정
BASE_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data"
CSV_PATH = os.path.join(BASE_PATH, "dev_test.csv")
IMAGE_DIR = os.path.join(BASE_PATH, "input_images")
OUTPUT_PATH = os.path.join(BASE_PATH, "submission_moondream.csv")

# 사용할 디바이스 설정 (GPU 우선 사용)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Moondream1 모델 로드 (1.6B parameters, Apache 2.0, 2023년 11월 공개)
# 대회 규칙을 모두 만족합니다.
moondream_model_id = "vikhyatk/moondream1"
moondream_model = AutoModelForCausalLM.from_pretrained(moondream_model_id, trust_remote_code=True).to(device)
moondream_tokenizer = AutoTokenizer.from_pretrained(moondream_model_id)

# 문장 유사도 비교를 위한 Sentence Transformer 모델 로드
# 작고 빠른 모델을 사용합니다.
similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# --- 2. 데이터 로드 ---

try:
    df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_PATH}")
    exit()

results = []
ANSWER_MAP = ['A', 'B', 'C', 'D']

# --- 3. 모델 추론 ---

print("Starting inference with Moondream1...")
with torch.no_grad():
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            test_id = row['ID']
            img_filename = row['img_path'].split('/')[-1]
            img_path = os.path.join(IMAGE_DIR, img_filename)
            
            question = row['Question']
            choices = [row['A'], row['B'], row['C'], row['D']]

            image = Image.open(img_path)
            # 이미지를 Moondream 모델 입력 형식으로 인코딩
            image_embeds = moondream_model.encode_image(image)

            # 1. 답변 생성 (by Moondream1)
            # 모델에 이미지와 질문을 함께 입력하여 답변을 생성합니다.
            generated_answer = moondream_model.answer_question(image_embeds=image_embeds, question=question, tokenizer=moondream_tokenizer)

            # 2. 선택지 매칭 (by Sentence Transformer)
            # 생성된 답변과 4개의 선택지 간의 유사도를 계산합니다.
            generated_answer_embedding = similarity_model.encode(generated_answer, convert_to_tensor=True)
            choice_embeddings = similarity_model.encode(choices, convert_to_tensor=True)

            # 코사인 유사도 계산
            cosine_scores = util.pytorch_cos_sim(generated_answer_embedding, choice_embeddings)
            
            # 가장 높은 점수를 가진 선택지의 인덱스 찾기
            prediction_index = torch.argmax(cosine_scores).item()
            
            # 인덱스를 'A', 'B', 'C', 'D'로 변환
            final_answer = ANSWER_MAP[prediction_index]
            
            results.append({'ID': test_id, 'answer': final_answer})

        except Exception as e:
            print(f"Error processing row {index} (ID: {row.get('ID', 'N/A')}): {e}")
            results.append({'ID': row.get('ID', 'N/A'), 'answer': 'E'}) # 에러 발생 시 'E'로 표기

# --- 4. 제출 파일 생성 ---

print("Inference finished. Creating submission file...")

submission_df = pd.DataFrame(results)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"Submission file created successfully at: {OUTPUT_PATH}")
print("\n--- Top 5 predictions ---")
print(submission_df.head())
print("------------------------")

Using device: cuda


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Starting inference with Moondream1...


100%|██████████| 60/60 [02:01<00:00,  2.02s/it]

Inference finished. Creating submission file...
Submission file created successfully at: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission_moondream.csv

--- Top 5 predictions ---
         ID answer
0  TEST_000      B
1  TEST_001      B
2  TEST_002      D
3  TEST_003      D
4  TEST_004      B
------------------------


In [4]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in moondream_model.parameters())
trainable_params = sum(p.numel() for p in moondream_model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 1,857,482,608개 (1857.48M)
학습 가능한 파라미터: 1,857,482,608개 (1857.48M)


In [ ]:
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import os
from tqdm import tqdm

# --- 1. Setup: Paths and Models ---

# CSV 파일 및 이미지 디렉토리 경로 설정
BASE_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2"
CSV_PATH = os.path.join(BASE_PATH, "test.csv")
IMAGE_DIR = os.path.join(BASE_PATH, "test_input_images")
# 사용자 요청에 따라 출력 파일 경로를 sample_submission.csv와 동일하게 설정
OUTPUT_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/clip_large.csv"

# 장치 설정 (GPU 사용 가능 시 GPU 사용)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 중인 장치: {device}")

# CLIP 모델 및 프로세서 로드 (대형 모델 사용)
# 'openai/clip-vit-large-patch14'는 일반적인 대형 CLIP 모델입니다.
model_id = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(model_id).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_id)

# --- 2. Data Loading ---

try:
    df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"오류: CSV 파일이 {CSV_PATH}에서 발견되지 않았습니다.")
    exit()

results = []
ANSWER_MAP = ['A', 'B', 'C', 'D']

# --- 3. Model Inference ---

print("CLIP Large를 사용한 추론을 시작합니다...")
with torch.no_grad(): # 추론을 위해 기울기 계산 비활성화
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            test_id = row['ID']
            img_filename = row['img_path'].split('/')[-1]
            img_path = os.path.join(IMAGE_DIR, img_filename)
            
            question = row['Question']
            choices = [str(row['A']), str(row['B']), str(row['C']), str(row['D'])]

            image = Image.open(img_path).convert("RGB")
            
            # CLIP 기반 VQA를 위해 질문과 각 답변 선택지를 결합하여 텍스트 후보를 구성합니다.
            # 그런 다음 어떤 결합된 텍스트가 이미지와 가장 잘 일치하는지 찾습니다.
            candidate_texts = [f"Question: {question} Answer: {c}" for c in choices]

            # CLIP을 위한 입력 처리: 이미지 및 모든 텍스트 후보
            # padding=True는 모든 텍스트 입력이 동일한 길이로 채워지도록 보장합니다.
            inputs = processor(text=candidate_texts, images=image, return_tensors="pt", padding=True).to(device)
            
            # CLIP 모델에서 이미지 및 텍스트 특징을 가져옵니다.
            outputs = model(**inputs)
            image_features = outputs.image_embeds # 입력 이미지에 대한 임베딩
            text_features = outputs.text_embeds   # 각 텍스트 후보에 대한 임베딩

            # 코사인 유사도 계산을 위해 특징을 정규화합니다.
            # 정규화는 정확한 코사인 유사도를 위해 중요합니다.
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

            # 이미지 특징과 각 텍스트 특징 간의 코사인 유사도를 계산합니다.
            # 결과는 이미지와 텍스트 후보 중 하나 사이의 유사도 점수인 텐서가 됩니다.
            cosine_scores = (image_features @ text_features.T).squeeze(0) # 1D 텐서를 얻기 위해 squeeze

            # 가장 높은 유사도 점수를 가진 선택지의 인덱스를 찾습니다.
            prediction_index = torch.argmax(cosine_scores).item()
            
            # 인덱스를 미리 정의된 맵을 사용하여 'A', 'B', 'C', 또는 'D'로 변환합니다.
            final_answer = ANSWER_MAP[prediction_index]
            
            results.append({'ID': test_id, 'answer': final_answer})

        except Exception as e:
            print(f"행 {index} (ID: {row.get('ID', 'N/A')}) 처리 중 오류 발생: {e}")
            # 오류 발생 시 답변을 'E'로 표시합니다.
            results.append({'ID': row.get('ID', 'N/A'), 'answer': 'E'})

# --- 4. Create Submission File ---

print("추론이 완료되었습니다. 제출 파일을 생성합니다...")

submission_df = pd.DataFrame(results)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"제출 파일이 {OUTPUT_PATH}에 성공적으로 생성되었습니다.")
print("\n--- 상위 5개 예측 ---")
print(submission_df.head())
print("------------------------")


사용 중인 장치: cuda
CLIP Large를 사용한 추론을 시작합니다...


100%|██████████| 852/852 [00:29<00:00, 29.13it/s]

추론이 완료되었습니다. 제출 파일을 생성합니다...
제출 파일이 /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/clip_large.csv에 성공적으로 생성되었습니다.

--- 상위 5개 예측 ---
         ID answer
0  TEST_000      B
1  TEST_001      A
2  TEST_002      B
3  TEST_003      C
4  TEST_004      C
------------------------
